<a href="https://colab.research.google.com/github/Jahnavi-CH/Pyspark-Learning/blob/main/Mini_assignment_1_Jahnavi_C_H.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 1: Write a function that removes duplicate rows, ensures correct data types for numerical and date columns and converts all ‘yes’/ ‘no’ type fields into 1/0 format.

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Mini assignment 1").getOrCreate()
df = spark.read.csv("sample_data/Lung Cancer.csv", header=True, inferSchema=True)
df.show()
df.printSchema()

+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
| id| age|gender|    country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|
+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|  1|64.0|  Male|     Sweden|    2016-04-05|     Stage I|           Yes|Passive Smoker|29.4|              199|           0|     0|        1|           0|  Chemotherapy|        2017-09-10|       0|
|  2|50.0|Female|Netherlands|    2023-04-20|   Stage III|           Yes|Passive Smoker|41.2|              280|           1|     1|        0|           0|       Surgery|        2024-06-17|       1|
|  3|65.0|Femal

In [2]:
print("Original count: ", df.count())
df = df.dropDuplicates()
print("After removing duplicates: ", df.count())
#df.show()


Original count:  430273
After removing duplicates:  430273


In [3]:
from pyspark.sql.functions import when, col

def clean_lung_cancer_data(dataframe):
    # 1. Remove duplicate rows
    print("Original row count: ", dataframe.count())
    cleaned_df = dataframe.dropDuplicates()
    print("Row count after removing duplicates: ", cleaned_df.count())

    # 2. Ensure correct data types for numerical and date columns
    # Based on printSchema() in previous cell, 'diagnosis_date' and 'end_treatment_date' are already date types.
    # Numerical columns (age, bmi, cholesterol_level, hypertension, asthma, cirrhosis, other_cancer, survived)
    # also appear to have correct types (double, int) as inferred by Spark.
    # No explicit casting is needed for these based on the current schema.

    # 3. Convert 'yes'/'no' type fields into 1/0 format
    # Assuming 'family_history' is the target column based on df.show()
    cleaned_df = cleaned_df.withColumn(
        "family_history",
        when(col("family_history") == "Yes", 1).otherwise(0)
    )

    return cleaned_df

In [4]:
# Apply the cleaning function to the DataFrame
cleaned_df = clean_lung_cancer_data(df)

# Display the schema and some data to verify changes
print("Schema after cleaning:")
cleaned_df.printSchema()

print("Data after cleaning (first 5 rows):")
cleaned_df.show(5)

Original row count:  430273
Row count after removing duplicates:  430273
Schema after cleaning:
root
 |-- id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- diagnosis_date: date (nullable = true)
 |-- cancer_stage: string (nullable = true)
 |-- family_history: integer (nullable = false)
 |-- smoking_status: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- cholesterol_level: integer (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- asthma: integer (nullable = true)
 |-- cirrhosis: integer (nullable = true)
 |-- other_cancer: integer (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- end_treatment_date: date (nullable = true)
 |-- survived: integer (nullable = true)

Data after cleaning (first 5 rows):
+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------

Task 2: Write a function that adds a new column, treatment_duration_days, which calculates the number of days between the diagnosis and the end of treatment. Then, return the average treatment duration for each treatment type.

In [5]:
from pyspark.sql.functions import datediff, col,avg

df_new = cleaned_df.withColumn(
    "treatment_duration_days",
    datediff(
        col('end_treatment_date'),
        col('diagnosis_date')
    )
)

In [6]:
df_new.show()

+----+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------------------+
|  id| age|gender|       country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|treatment_duration_days|
+----+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------------------+
|  72|54.0|Female|Czech Republic|    2017-06-30|    Stage IV|             0|Passive Smoker|24.0|              160|           1|     0|        0|           1|       Surgery|        2019-04-06|       0|                    645|
| 190|34.0|  Male|       Estonia|    2015-02-13|     Stage I|             1| Former Smoker|38.8|    

In [7]:
result = df_new.groupBy("treatment_type").agg(avg("treatment_duration_days").alias("avg_treatment_duration"))

In [8]:
result.show()

+--------------+----------------------+
|treatment_type|avg_treatment_duration|
+--------------+----------------------+
|     Radiation|     458.2585218505391|
|  Chemotherapy|    458.53765312641787|
|      Combined|     457.5999647292506|
|       Surgery|      457.637479671019|
+--------------+----------------------+



Task 3: Write a function that returns the smoking_status group with the highest survival rate.

In [9]:
df_new.show(5)

+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------------------+
| id| age|gender|       country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|treatment_duration_days|
+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------------------+
| 72|54.0|Female|Czech Republic|    2017-06-30|    Stage IV|             0|Passive Smoker|24.0|              160|           1|     0|        0|           1|       Surgery|        2019-04-06|       0|                    645|
|190|34.0|  Male|       Estonia|    2015-02-13|     Stage I|             1| Former Smoker|38.8|         

In [10]:
unique_smoking_status = [row[0] for row in df_new.select('smoking_status').distinct().collect()]
print(unique_smoking_status)

['Never Smoked', 'Former Smoker', 'Current Smoker', 'Passive Smoker']


In [11]:
from pyspark.sql.functions import avg

highest_survived = df_new.groupBy("smoking_status").agg(avg("survived").alias("survival_rate"))

To find the smoking status with the highest survival rate, you need to sort your highest_survived DataFrame by survival_rate in descending order and then select the top entry.

The smoking status group with the highest survival rate is 'Never Smoked' with a survival rate of 0.2209. This result was obtained by ordering the survival rates in descending order and selecting the top entry.

In [12]:
highest_survival_group = highest_survived.orderBy(col('survival_rate').desc()).first()
print(f"The smoking status group with the highest survival rate is: {highest_survival_group['smoking_status']} with a rate of \
{highest_survival_group['survival_rate']:.4f}")

The smoking status group with the highest survival rate is: Never Smoked with a rate of 0.2210


Task 4: Write a function that returns the top three countries with the highest percentage of patients diagnosed in Stage IV.  



In [13]:
df_new.show(5)

+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------------------+
| id| age|gender|       country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|treatment_duration_days|
+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------------------+
| 72|54.0|Female|Czech Republic|    2017-06-30|    Stage IV|             0|Passive Smoker|24.0|              160|           1|     0|        0|           1|       Surgery|        2019-04-06|       0|                    645|
|190|34.0|  Male|       Estonia|    2015-02-13|     Stage I|             1| Former Smoker|38.8|         

In [14]:
from pyspark.sql.functions import col
# Filter the DataFrame to get only patients diagnosed in Stage IV
stage_4_cancer_df = df_new.filter(col("cancer_stage") == "Stage IV")

In [15]:
stage_4_cancer_df.show()

+-----+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------------------+
|   id| age|gender|       country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|treatment_duration_days|
+-----+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------------------+
|   72|54.0|Female|Czech Republic|    2017-06-30|    Stage IV|             0|Passive Smoker|24.0|              160|           1|     0|        0|           1|       Surgery|        2019-04-06|       0|                    645|
|  404|57.0|  Male|       Austria|    2016-07-29|    Stage IV|             1|Current Smoker|40.9

In [16]:
from pyspark.sql.functions import count, col

stage_4_patients_per_country = stage_4_cancer_df.groupBy("country").agg(
    count("id").alias("stage_4_patients_count")
)

In [17]:
stage_4_patients_per_country.show()

+--------------+----------------------+
|       country|stage_4_patients_count|
+--------------+----------------------+
|        Sweden|                  3924|
|       Germany|                  3898|
|        France|                  4087|
|        Greece|                  4075|
|      Slovakia|                  3961|
|       Belgium|                  4006|
|       Finland|                  3936|
|         Malta|                  4072|
|       Croatia|                  3986|
|         Italy|                  4068|
|     Lithuania|                  4006|
|         Spain|                  4030|
|       Denmark|                  4026|
|       Ireland|                  4075|
|        Cyprus|                  3981|
|       Estonia|                  3832|
|        Latvia|                  3964|
|Czech Republic|                  4122|
|      Slovenia|                  4076|
|    Luxembourg|                  3948|
+--------------+----------------------+
only showing top 20 rows


In [18]:
from pyspark.sql.functions import count, lit

# Count total patients per country
total_patients_per_country = df_new.groupBy("country").agg(count("id").alias("total_patients"))

In [19]:
total_patients_per_country.show()

+--------------+--------------+
|       country|total_patients|
+--------------+--------------+
|        Sweden|         15999|
|       Germany|         15879|
|        France|         16059|
|        Greece|         15971|
|      Slovakia|         15947|
|       Belgium|         16009|
|       Finland|         15938|
|         Malta|         16097|
|       Croatia|         16088|
|         Italy|         15961|
|     Lithuania|         15822|
|         Spain|         15929|
|       Denmark|         15881|
|       Ireland|         16237|
|        Cyprus|         15748|
|       Estonia|         15885|
|        Latvia|         15843|
|Czech Republic|         16035|
|      Slovenia|         15965|
|    Luxembourg|         15835|
+--------------+--------------+
only showing top 20 rows


In [20]:
# Join the two DataFrames and calculate the percentage
percentage_stage_4_per_country = total_patients_per_country.join(
    stage_4_patients_per_country, "country", "left_outer"
).withColumn(
    "stage_4_percentage",
    (col("stage_4_patients_count") / col("total_patients")) * 100
).fillna(0, subset=["stage_4_patients_count", "stage_4_percentage"]) # Fill NaN with 0 for countries with no Stage IV patients

# Order by percentage and show the top three
percentage_stage_4_per_country.orderBy(col("stage_4_percentage").desc()).show(3)

+--------------+--------------+----------------------+------------------+
|       country|total_patients|stage_4_patients_count|stage_4_percentage|
+--------------+--------------+----------------------+------------------+
|Czech Republic|         16035|                  4122|25.706267539756784|
|      Slovenia|         15965|                  4076|25.530848731600376|
|        Greece|         15971|                  4075|25.514995930123348|
+--------------+--------------+----------------------+------------------+
only showing top 3 rows


Let's break down the code in the last cell step by step:

1.  **`percentage_stage_4_per_country = total_patients_per_country.join(stage_4_patients_per_country, "country", "left_outer")`**
    *   This line performs a **`join`** operation between two DataFrames:
        *   `total_patients_per_country`: Contains the total number of patients for each country.
        *   `stage_4_patients_per_country`: Contains the count of Stage IV patients for each country.
    *   `"country"`: Specifies that the join should happen on the common `country` column.
    *   `"left_outer"`: This is the type of join. It means all rows from the `total_patients_per_country` (left DataFrame) will be included. If a country from the left DataFrame doesn't have a matching entry in `stage_4_patients_per_country`, the columns from `stage_4_patients_per_country` will have `null` values for that row.

2.  **`.withColumn("stage_4_percentage", (col("stage_4_patients_count") / col("total_patients")) * 100)`**
    *   This part adds a **new column** named `"stage_4_percentage"` to the joined DataFrame.
    *   The value for this new column is calculated as: `(Stage IV patients count / total patients count) * 100`. This gives us the percentage of Stage IV patients for each country.

3.  **`.fillna(0, subset=["stage_4_patients_count", "stage_4_percentage"])`**
    *   After the `left_outer` join, if a country had no Stage IV patients, the `stage_4_patients_count` and consequently `stage_4_percentage` columns would be `null`.
    *   This **`fillna`** operation replaces any `null` values in the specified `subset` of columns (`"stage_4_patients_count"`, `"stage_4_percentage"`) with `0`.

4.  **`.orderBy(col("stage_4_percentage").desc())`**
    *   This **`orderBy`** clause sorts the DataFrame based on the `"stage_4_percentage"` column.
    *   `.desc()` specifies that the sorting should be in descending order, meaning countries with the highest percentage will appear first.

5.  **`.show(3)`**
    *   Finally, this displays the first `3` rows of the resulting DataFrame, which, after the sorting, will be the top three countries with the highest percentage of Stage IV patients.

### Alternative Method: Conditional Aggregation

This approach calculates the total number of patients and the number of Stage IV patients for each country in a single aggregation step, avoiding the need for a separate join operation.

In [21]:
from pyspark.sql.functions import col, sum, when

# Calculate total patients and stage 4 patients per country in one step
summary_df = df_new.groupBy("country").agg(
    count("id").alias("total_patients"),
    sum(when(col("cancer_stage") == "Stage IV", 1).otherwise(0)).alias("stage_4_patients_count")
)

# Calculate the percentage
percentage_stage_4_per_country_alt = summary_df.withColumn(
    "stage_4_percentage",
    (col("stage_4_patients_count") / col("total_patients")) * 100
).fillna(0, subset=["stage_4_patients_count", "stage_4_percentage"]) # Fill NaN with 0 for robustness

# Order by percentage and show the top three
percentage_stage_4_per_country_alt.orderBy(col("stage_4_percentage").desc()).show(3)

+--------------+--------------+----------------------+------------------+
|       country|total_patients|stage_4_patients_count|stage_4_percentage|
+--------------+--------------+----------------------+------------------+
|Czech Republic|         16035|                  4122|25.706267539756784|
|      Slovenia|         15965|                  4076|25.530848731600376|
|        Greece|         15971|                  4075|25.514995930123348|
+--------------+--------------+----------------------+------------------+
only showing top 3 rows


Task 5: Write a function that filters patients who:  

Are male  

Diagnosed in Stage III or IV  

Have a family history of cancer  

Are current smokers  

Have a BMI > 30  

Survived

Return the average age and the percentage of these patients who had hypertension.



In [22]:
df_new.show(5)

+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------------------+
| id| age|gender|       country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|treatment_duration_days|
+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------------------+
| 72|54.0|Female|Czech Republic|    2017-06-30|    Stage IV|             0|Passive Smoker|24.0|              160|           1|     0|        0|           1|       Surgery|        2019-04-06|       0|                    645|
|190|34.0|  Male|       Estonia|    2015-02-13|     Stage I|             1| Former Smoker|38.8|         

In [23]:
from pyspark.sql.functions import col

df_filtered = df_new.filter(
    (col('gender') == 'Male') &
    ((col('cancer_stage') == 'Stage III') | (col('cancer_stage') == 'Stage IV')) &
    (col('family_history') == 1) &
    (col('smoking_status') == 'Current Smoker') &
    (col('bmi') > 30) &
    (col('survived') == 1)
)

In [27]:
df_filtered.show(5)

+------+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------------------+
|    id| age|gender|    country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|treatment_duration_days|
+------+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+-----------------------+
| 18230|63.0|  Male|   Bulgaria|    2023-02-21|   Stage III|             1|Current Smoker|32.7|              268|           1|     0|        0|           0|     Radiation|        2023-10-01|       1|                    222|
| 82700|63.0|  Male|Netherlands|    2021-10-01|    Stage IV|             1|Current Smoker|30.8|         

In [28]:
from pyspark.sql.functions import avg, col

# Calculate the average age and the percentage of patients with hypertension
result_task5 = df_filtered.agg(
    avg(col('age')).alias("Average Age"),
    (avg(col('hypertension')) * 100).alias("Percentage Hypertension")
)

# Show the result
result_task5.show()

+----------------+-----------------------+
|     Average Age|Percentage Hypertension|
+----------------+-----------------------+
|55.5369170984456|      75.38860103626943|
+----------------+-----------------------+

